# PAIMANA-Style Infrastructure Project Monitoring: Exploratory Data Analysis (EDA)

This notebook explores the synthetic multi-snapshot project-monitoring dataset (`projects_snapshot.csv`).
It inspects data quality, distributional properties, risk signatures, target class balance, and candidate features for future predictive modeling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (9, 5)
print("Libraries loaded successfully.")

## 1. Dataset Loading & Dimension Verification

In [ ]:
data_path = config.get_active_dataset_path()
df = pd.read_csv(data_path)
print(f"Total Snapshots (Rows) : {len(df):,}")
print(f"Unique Projects        : {df['project_id'].nunique():,}")
print(f"Average Snaps / Project: {len(df) / df['project_id'].nunique():.2f}")
df.head()

## 2. Missing Values and Duplicate Integrity Check

In [ ]:
null_counts = df.isnull().sum()
duplicates = df.duplicated(subset=["project_id", "snapshot_month"]).sum()

print(f"Total Missing Values Across Dataset : {null_counts.sum()}")
print(f"Duplicate (project_id + snapshot)  : {duplicates}")
assert null_counts.sum() == 0, "Missing values detected!"
assert duplicates == 0, "Duplicate snapshots detected!"
print("\nIntegrity Check: PASSED")

## 3. Distribution of Numerical Attributes

In [ ]:
numeric_cols = [
    "original_cost_cr", "planned_duration_months", "elapsed_months",
    "physical_progress_pct", "financial_progress_pct", "expenditure_cr",
    "milestones_total", "milestones_delayed"
]
df[numeric_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["original_cost_cr"], kde=True, log_scale=True, color="#1f77b4", ax=axes[0])
axes[0].set_title("Original Cost (₹ Cr, Log Scale)")
axes[0].axvline(df["original_cost_cr"].median(), color="red", linestyle="--", label=f"Median: ₹{df['original_cost_cr'].median():.1f} Cr")
axes[0].legend()

sns.histplot(df["planned_duration_months"], kde=True, color="#2ca02c", alpha=0.6, label="Planned", ax=axes[1])
sns.histplot(df["actual_duration_months"], kde=True, color="#d62728", alpha=0.5, label="Actual", ax=axes[1])
axes[1].set_title("Planned vs Actual Duration (Months)")
axes[1].legend()
plt.show()

## 4. Target Class Balance Analysis

In [ ]:
cost_balance = df["cost_overrun"].value_counts(normalize=True) * 100
time_balance = df["time_overrun"].value_counts(normalize=True) * 100

print("Cost Overrun Balance:")
for label, pct in cost_balance.items():
    print(f"  Class {label}: {pct:.1f}%")

print("\nTime Overrun Balance:")
for label, pct in time_balance.items():
    print(f"  Class {label}: {pct:.1f}%")

pd.crosstab(df["cost_overrun"], df["time_overrun"], margins=True, normalize="all").round(3) * 100

## 5. Risk Signals & Feature Relationships

In [ ]:
# Compute risk indicators
df["progress_gap"] = df["financial_progress_pct"] - df["physical_progress_pct"]
df["schedule_utilization_pct"] = (df["elapsed_months"] / df["planned_duration_months"]) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sample_subset = df.sample(min(2000, len(df)), random_state=42)
sns.scatterplot(
    data=sample_subset,
    x="physical_progress_pct",
    y="financial_progress_pct",
    hue="cost_overrun",
    palette={0: "#2ca02c", 1: "#d62728"},
    alpha=0.6,
    ax=ax
)
ax.plot([0, 100], [0, 100], "--", color="gray", label="1:1 Parity")
ax.set_title("Physical vs. Financial Progress by Cost Overrun")
ax.legend()
plt.show()

## 6. Correlation Analysis

In [ ]:
corr_cols = [
    "original_cost_cr", "planned_duration_months", "elapsed_months",
    "physical_progress_pct", "financial_progress_pct", "progress_gap",
    "milestones_total", "milestones_delayed", "cost_overrun", "time_overrun"
]
plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap of Candidate Signals & Targets")
plt.show()